# 12_GFS_Forecast_Extraction_v2.ipynb
Enter a city name, geocode it with OpenStreetMap Nominatim, then download NOAA GFS forecast.

In [ ]:
!pip -q install geopy xarray siphon pandas netCDF4 requests

import pandas as pd
import xarray as xr
from geopy.geocoders import Nominatim
from google.colab import files

city=input("Enter City Name: ").strip()

geolocator=Nominatim(user_agent="heatwave-flood-prediction")
location=geolocator.geocode(city)

if location is None:
    raise ValueError("City not found.")

lat=location.latitude
lon=location.longitude

print(f"City      : {city}")
print(f"Latitude  : {lat}")
print(f"Longitude : {lon}")

url="https://nomads.ncep.noaa.gov/dods/gfs_0p25/gfs_latest"
print("Opening NOAA GFS...")
ds=xr.open_dataset(url)

point=ds.sel(lat=lat,lon=lon,method="nearest")
subset=point.isel(time=slice(0,73))

rows=[]
for i in range(len(subset.time)):
    row={
        "city":city,
        "forecast_time":str(subset.time.values[i]),
        "latitude":lat,
        "longitude":lon
    }
    mapping={
        "tmp2m":"temperature_k",
        "pressfc":"pressure_pa",
        "ugrd10m":"u_wind",
        "vgrd10m":"v_wind",
        "rh2m":"relative_humidity",
        "apcpsfc":"precipitation"
    }
    for var,new in mapping.items():
        row[new]=float(subset[var].values[i]) if var in subset.variables else None
    rows.append(row)

gfs=pd.DataFrame(rows)
if "temperature_k" in gfs.columns:
    gfs["temperature_c"]=gfs["temperature_k"]-273.15

gfs.to_csv("gfs_features.csv",index=False)
print(gfs.head())
files.download("gfs_features.csv")
